# Logistics EDA: simulated warehouse deliveries
One row per order; UTC snapshot; intentionally confounded carrier allocation.
Run all cells in this notebook. DataFrames and plots are displayed inline; data are generated in memory.

## 1. Imports and analytical contract

In [ ]:
from IPython.display import display
import importlib.metadata
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

SEED = 42
SNAPSHOT = pd.Timestamp("2026-04-02 00:00:00", tz="UTC")
sns.set_theme(style="whitegrid", palette="colorblind", context="notebook")

## 2. Simulate source systems, including realistic quality problems

In [ ]:
def simulate():
    rng = np.random.default_rng(SEED)
    records = []
    order_id = 0
    # This synthetic data generator is for demonstration, not a model of a real logistics system.
    for day in pd.date_range("2026-01-01", periods=90, tz="UTC"):
        volume = int(rng.poisson(65 + 15 * (day.dayofweek == 0)))
        shared_day_delay = rng.normal(0, 2)
        for _ in range(volume):
            order_id += 1
            lane = rng.choice(["Local", "Regional"])
            carrier = "A" if rng.random() < (0.85 if lane == "Regional" else 0.15) else "B"
            distance = max(10, rng.normal(320 if lane == "Regional" else 55, 12))
            units = int(rng.integers(1, 9))
            released = day + pd.Timedelta(minutes=int(rng.integers(0, 1440)))
            lead_h = max(1, (49 if lane == "Regional" else 19)
                         + (0 if carrier == "A" else 3)
                         + 0.018 * (distance - (320 if lane == "Regional" else 55))
                         + 0.15 * units + 0.07 * (volume - 65)
                         + shared_day_delay + rng.normal(0, 3))
            if rng.random() < 0.015:
                lead_h += rng.uniform(15, 35)
            delivered = released + pd.Timedelta(hours=lead_h)
            status = "delivered" if delivered <= SNAPSHOT else "in_transit"
            records.append([f"O{order_id:06d}", released, lane, carrier,
                            distance, units, delivered if status == "delivered" else pd.NaT, status])
    raw = pd.DataFrame(records, columns=["order_id", "released_at", "lane", "carrier",
                                        "distance_km", "units", "delivered_at", "status"])
    # Inject data-quality issues so the cleaning steps have examples to find.
    done = raw.index[raw.status.eq("delivered")].to_numpy()
    bad = rng.choice(done, 37, replace=False)
    raw.loc[bad[:25], "delivered_at"] = pd.NaT
    raw.loc[bad[25:], "delivered_at"] = raw.loc[bad[25:], "released_at"] - pd.Timedelta(hours=2)
    raw.loc[rng.choice(raw.index, 80, replace=False), "distance_km"] = np.nan
    raw.loc[rng.choice(raw.index, 60, replace=False), "carrier"] = " a "
    original = pd.Series([record[3] for record in records], index=raw.index)
    changed = raw.carrier.eq(" a ")
    raw.loc[changed, "carrier"] = original[changed].str.lower().map(lambda value: f" {value} ")
    raw = pd.concat([raw, raw.sample(30, random_state=SEED)], ignore_index=True)
    lanes = pd.DataFrame({"lane": ["Local", "Regional"], "sla_h": [24, 48]})
    return raw, lanes

raw, lanes = simulate()
display(raw.head(10))
display(lanes)

## 3. Read, profile, validate grain, clean, and join

In [ ]:
display(pd.DataFrame({"rows": [len(raw)], "columns": [raw.columns.size],
                       "exact_duplicate_rows": [int(raw.duplicated().sum())]}))
display(raw.describe(include="all").T)
display(raw.isna().mean().rename("missing_fraction").to_frame())

audit = {"raw_rows": len(raw), "exact_duplicate_rows": int(raw.duplicated().sum())}
d = raw.drop_duplicates().copy()
d["carrier"] = d.carrier.str.strip().str.upper()
for col in ["released_at", "delivered_at"]:
    d[col] = pd.to_datetime(d[col], utc=True, errors="coerce", format="mixed")
for col in ["distance_km", "units"]:
    d[col] = pd.to_numeric(d[col], errors="coerce")

display(d.head(10))
assert d.released_at.notna().all()
assert d.carrier.isin(["A", "B"]).all()
assert d.status.isin(["delivered", "in_transit"]).all()
assert d.units.ge(1).all()

bad_time = d.delivered_at.lt(d.released_at) | d.delivered_at.gt(SNAPSHOT)
audit["invalid_delivery_times"] = int(bad_time.sum())
d["invalid_delivery_time"] = bad_time
d.loc[bad_time, "delivered_at"] = pd.NaT
d.loc[d.distance_km.le(0), "distance_km"] = np.nan
d["distance_missing"] = d.distance_km.isna()
audit["missing_distance"] = int(d.distance_missing.sum())

d = d.merge(lanes, on="lane", how="left", validate="many_to_one", indicator=True)
assert d["_merge"].eq("both").all(), "Unknown lane"
d = d.drop(columns="_merge")
display(d.head(10))

## 4. Define metrics, eligibility, missing outcomes, and time features

In [ ]:
d["due_at"] = d.released_at + pd.to_timedelta(d.sla_h, unit="h")
d["lead_h"] = (d.delivered_at - d.released_at).dt.total_seconds() / 3600
d["day"] = d.released_at.dt.floor("D")
d["weekday"] = d.released_at.dt.day_name()
d["day_index"] = (d.day - d.day.min()).dt.days
d["daily_orders"] = d.groupby("day").order_id.transform("size")
d["matured"] = d.due_at.le(SNAPSHOT)
known_delivery = d.status.eq("delivered") & d.delivered_at.notna()
known_open_late = d.status.eq("in_transit") & d.matured
d["on_time"] = pd.Series(pd.NA, index=d.index, dtype="boolean")
eligible_known = d.matured & known_delivery
d.loc[eligible_known, "on_time"] = d.loc[eligible_known, "delivered_at"].le(d.loc[eligible_known, "due_at"])
d.loc[known_open_late, "on_time"] = False
d["late"] = ~d.on_time
eligible = d[d.on_time.notna()].copy()
eligible["on_time"] = eligible.on_time.astype(bool)
eligible["late"] = eligible.late.astype(bool)
completed = d[d.lead_h.notna()].copy()

audit.update(unique_orders=len(d), matured_orders=int(d.matured.sum()),
             known_matured_outcomes=len(eligible),
             unknown_matured_outcomes=int((d.matured & d.on_time.isna()).sum()),
             not_yet_due=int((~d.matured).sum()),
             valid_completed_lead_times=len(completed))
display(completed.head(10))
display(pd.Series(audit, name="count").to_frame())

## 5. Summary statistics and correct denominators

In [ ]:
summary = completed.groupby(["carrier", "lane"]).lead_h.agg(
    n="size", mean_h="mean", median_h="median", p90_h=lambda x: x.quantile(.90), sd_h="std")
display(summary.style.format({"mean_h": "{:.1f}", "median_h": "{:.1f}", "p90_h": "{:.1f}", "sd_h": "{:.1f}"}))

kpi = eligible.groupby(["carrier", "lane"]).agg(
    n=("order_id", "size"), on_time=("on_time", "sum"), late=("late", "sum"))
kpi["on_time_rate"] = kpi.on_time / kpi.n
display(kpi.style.format({"on_time_rate": "{:.1%}"}))

overall_rate = eligible.on_time.mean()
matured_n = int(d.matured.sum())
unknown_n = audit["unknown_matured_outcomes"]
success_n = int(eligible.on_time.sum())
bounds = (success_n / matured_n, (success_n + unknown_n) / matured_n)
display(pd.DataFrame({"known_matured_on_time_rate": [overall_rate],
                      "lower_bound_unknown_all_late": [bounds[0]],
                      "upper_bound_unknown_all_on_time": [bounds[1]]}).style.format("{:.1%}"))

## 6. Daily trends (e.g. daily on-time rate)

In [ ]:
daily = eligible.groupby("day").agg(n=("order_id", "size"), success=("on_time", "sum"))
daily = daily.reindex(pd.date_range(d.day.min(), d.day.max(), freq="D"))
daily["rate"] = daily.success / daily.n
daily["rolling_7d_rate"] = daily.success.rolling(7, min_periods=7).sum() / daily.n.rolling(7, min_periods=7).sum()
daily.index.name = "day"
display(daily.head(10).style.format({"rate": "{:.1%}", "rolling_7d_rate": "{:.1%}"}))
display(daily.tail(10).style.format({"rate": "{:.1%}", "rolling_7d_rate": "{:.1%}"}))

## 7. Distributions and outliers: flag for investigation, retain valid tails

In [ ]:
q1 = completed.groupby("lane").lead_h.transform(lambda x: x.quantile(.25))
q3 = completed.groupby("lane").lead_h.transform(lambda x: x.quantile(.75))
completed["iqr_flag"] = (completed.lead_h < q1 - 1.5 * (q3-q1)) | (completed.lead_h > q3 + 1.5 * (q3-q1))
outlier_review = completed.loc[completed.iqr_flag].sort_values("lead_h", ascending=False)
display(pd.DataFrame({"flagged_orders": [len(outlier_review)], "completed_orders": [len(completed)]}))
display(outlier_review.head(20))

## 8. Relationships and confounding: raw versus lane-standardized rates

In [ ]:
raw_carrier = eligible.groupby("carrier").on_time.mean()
stratified = eligible.groupby(["carrier", "lane"]).on_time.mean().unstack()
weights = eligible.lane.value_counts(normalize=True).reindex(stratified.columns)
assert stratified.notna().all().all(), "Standardization requires overlap"
standardized = stratified.mul(weights, axis=1).sum(axis=1)
comparison = pd.DataFrame({"raw_on_time_rate": raw_carrier,
                           "common_lane_mix_on_time_rate": standardized})
display(comparison.style.format("{:.1%}"))

pair = completed[["distance_km", "lead_h"]].dropna()
pearson = stats.pearsonr(pair.distance_km, pair.lead_h).statistic
spearman = stats.spearmanr(pair.distance_km, pair.lead_h).statistic
within_lane = completed.groupby("lane")[["distance_km", "lead_h"]].corr(method="spearman")
display(pd.DataFrame({"pearson_distance_lead": [pearson], "spearman_distance_lead": [spearman]},
                     index=["pooled association"]))
display(within_lane)

## 10. Inline plots: comparisons, distributions, trends, relationships

In [ ]:
from io import BytesIO
from IPython.display import Image, display

def display_figure(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=150)
    display(Image(data=buffer.getvalue()))
    plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
sns.ecdfplot(data=completed, x="lead_h", hue="lane", ax=axes[0, 0])
axes[0, 0].set(title="Completed orders: lead-time distribution", xlabel="Lead time (hours)", ylabel="Cumulative share")

sns.boxplot(data=completed, x="lane", y="lead_h", hue="carrier", ax=axes[0, 1], showfliers=False)
axes[0, 1].set(title="Carrier comparisons within each lane", xlabel="Lane", ylabel="Lead time (hours)")

axes[1, 0].plot(daily.index, daily.rate * 100, alpha=.35, label="Daily")
axes[1, 0].plot(daily.index, daily.rolling_7d_rate * 100, label="7-day pooled rate")
axes[1, 0].set(title="Service by release cohort (known, matured)", xlabel="Release date (UTC)", ylabel="On-time deliveries (%)", ylim=(0, 100))
axes[1, 0].legend()
date_locator = mdates.AutoDateLocator(minticks=4, maxticks=6)
axes[1, 0].xaxis.set_major_locator(date_locator)
axes[1, 0].xaxis.set_major_formatter(mdates.ConciseDateFormatter(date_locator))
comparison.mul(100).rename(columns={"raw_on_time_rate": "Raw", "common_lane_mix_on_time_rate": "Common lane mix"}).plot.bar(ax=axes[1, 1], rot=0)
axes[1, 1].set(title="Carrier ranking depends on work mix", xlabel="Carrier", ylabel="On-time deliveries (%)", ylim=(0, 100))
sns.despine(fig=fig)
display_figure(fig)

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
sns.scatterplot(data=completed.sample(min(1500, len(completed)), random_state=SEED), x="distance_km", y="lead_h", hue="lane", alpha=.35, s=15, ax=ax)
ax.set(title="Distance and lead time: route mix is a confounder", xlabel="Distance (km)", ylabel="Lead time (hours)")
display_figure(fig)

## 11. Interactive summary and reproducibility

In [ ]:
metrics = {"simulated": True, "snapshot_utc": str(SNAPSHOT), "seed": SEED,
           "unique_orders": len(d), "known_matured_orders": len(eligible),
           "on_time_rate": float(overall_rate),
           "missing_outcome_bounds": list(bounds), "pearson_distance_lead": float(pearson),
           "spearman_distance_lead": float(spearman)}
display(pd.Series(metrics, name="value").to_frame())
display(pd.Series(audit, name="count").to_frame())

try:
    import plotly.express as px
except ImportError:
    display(pd.DataFrame({"interactive_plot": ["Install plotly to display the interactive scatter plot."]}))
else:
    interactive = px.scatter(completed, x="distance_km", y="lead_h", color="lane", symbol="carrier",
                             hover_data=["order_id"], opacity=.35, template="plotly_white",
                             labels={"distance_km": "Distance (km)", "lead_h": "Lead time (hours)"},
                             title="Explore simulated order-level relationships")
    display(interactive)

versions = {"python": __import__("platform").python_version()}
for package in ["numpy", "pandas", "scipy", "matplotlib", "seaborn", "statsmodels", "plotly"]:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = "not installed"
display(pd.Series(versions, name="version").to_frame())